<a href="https://colab.research.google.com/github/VarshaSpoorthi-shree/infosys-franchise-ai/blob/main/authentication_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q streamlit pyjwt bcrypt pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 74.5 MB/s eta 0:00:00


In [2]:

import sqlite3
import bcrypt
import re
import os

DB_NAME = "users.db"

def get_connection():
    conn = sqlite3.connect(DB_NAME)
    conn.row_factory = sqlite3.Row
    return conn

def initialize_database():
    with get_connection() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS users (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                username TEXT NOT NULL UNIQUE COLLATE NOCASE,
                email TEXT NOT NULL UNIQUE COLLATE NOCASE,
                password_hash BLOB NOT NULL,
                security_question TEXT NOT NULL,
                security_answer_hash BLOB NOT NULL
            )
        """)

initialize_database()
print("Database initialized successfully!")


Database initialized successfully!


In [3]:

def validate_email(email):
    return bool(re.fullmatch(
        r"[A-Za-z]{2,}@[A-Za-z]{2,}(?:\.[A-Za-z]{2,})+",
        email.strip()
    ))

def validate_password(password):
    return (
        len(password) >= 8
        and re.search(r"[A-Z]", password)
        and re.search(r"[a-z]", password)
        and re.search(r"\d", password)
        and re.search(r"[^A-Za-z0-9]", password)
    )

print("Email and password validation functions are ready.")


Email and password validation functions are ready.


In [4]:
from google.colab import userdata
import os

for name in ["JWT_SECRET", "EMAIL_ADDRESS", "EMAIL_PASSWORD"]:
    value = userdata.get(name)
    if not value:
        raise ValueError(
            f"{name} is missing from Colab Secrets. "
            "Check the secret name and enable notebook access."
        )
    os.environ[name] = value

print("JWT and Gmail secrets loaded successfully.")

JWT and Gmail secrets loaded successfully.


In [5]:

from google.colab import userdata
import os

os.environ["JWT_SECRET"] = userdata.get("JWT_SECRET")

print("JWT secret loaded successfully.")


JWT secret loaded successfully.


In [6]:

from google.colab import userdata

sender_email = userdata.get("EMAIL_ADDRESS")
email_password = userdata.get("EMAIL_PASSWORD")

if sender_email and email_password:
    print("Gmail secrets are configured.")
else:
    print("Missing Gmail secrets. Check Colab Secrets.")


Gmail secrets are configured.


In [7]:
from google.colab import userdata
import os

for key in [
    "JWT_SECRET",
    "EMAIL_ADDRESS",
    "EMAIL_PASSWORD",
    "ADMIN_PASSWORD"
]:
    os.environ[key] = userdata.get(key)

print("Secrets loaded successfully.")

Secrets loaded successfully.


In [8]:

import secrets

print(secrets.token_hex(32))


3b8c33175ea8d30ada47ee7f2f37ccd95e00d04b3855fecfb4e2037b30efcc01


In [9]:
!cat streamlit.log

cat: streamlit.log: No such file or directory


In [12]:
from google.colab import userdata
import os
import subprocess
import time

# Load secrets
for name in ["JWT_SECRET", "EMAIL_ADDRESS", "EMAIL_PASSWORD"]:
    value = userdata.get(name)
    if not value or not value.strip():
        raise ValueError(f"{name} is missing or empty.")
    os.environ[name] = value.strip()

print("All secrets loaded.")

# Stop existing Streamlit processes
subprocess.run(
    ["pkill", "-f", "streamlit run /content/app.py"],
    capture_output=True
)
time.sleep(2)

# Start Streamlit with the loaded environment
with open("/content/streamlit.log", "w") as log:
    process = subprocess.Popen(
        [
            "streamlit", "run", "/content/app.py",
            "--server.port", "8501",
            "--server.headless", "true"
        ],
        env=os.environ.copy(),
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True
    )

print("Streamlit launched with PID:", process.pid)

All secrets loaded.
Streamlit launched with PID: 919


In [16]:

%%writefile app.py
import os
import re
import sqlite3
import secrets
import smtplib
import bcrypt
import jwt
import streamlit as st

from datetime import datetime, timedelta, timezone
from email.message import EmailMessage


# ============================================================
# 1. CONFIGURATION
# ============================================================

st.set_page_config(
    page_title="SecureAuth | Authentication Portal",
    page_icon="🔐",
    layout="wide",
    initial_sidebar_state="expanded"
)

DB_NAME = "users.db"
JWT_SECRET = os.environ.get("JWT_SECRET")
EMAIL_ADDRESS = os.environ.get("EMAIL_ADDRESS")
EMAIL_PASSWORD = os.environ.get("EMAIL_PASSWORD")
ADMIN_USERNAME = "admin"
ADMIN_PASSWORD = os.environ.get("ADMIN_PASSWORD", "")

SECURITY_QUESTIONS = [
    "What was the name of your first school?",
    "What is the name of your hometown?",
    "What was your childhood nickname?",
    "What is the name of your favourite teacher?"
]

if not JWT_SECRET:
    st.error("JWT_SECRET is missing. Configure it in Colab Secrets.")
    st.stop()


# ============================================================
# 2. LIGHT LAVENDER UI
# ============================================================

st.markdown("""
<style>
:root {
    --bg: #F7F5FF;
    --surface: #FFFFFF;
    --purple: #7651D6;
    --purple-dark: #5735B5;
    --purple-light: #EEE8FF;
    --text: #292044;
    --muted: #756D8B;
    --border: #E6DFF5;
}

.stApp {
    background:
        radial-gradient(ellipse at 0% 0%,
            rgba(196, 181, 253, .22), transparent 38%),
        #F7F5FF;
    color: var(--text);
}

[data-testid="stHeader"] {
    background: rgba(247, 245, 255, .92);
}

.block-container {
    max-width: 1180px;
    padding-top: 2rem;
    padding-bottom: 3rem;
}

h1, h2, h3, h4 {
    color: var(--text) !important;
    letter-spacing: -.4px;
}

p, li {
    color: #514967;
}

[data-testid="stCaptionContainer"] {
    color: var(--muted) !important;
}

/* SIDEBAR */

[data-testid="stSidebar"] {
    background: linear-gradient(180deg, #F0EBFF, #F8F6FF);
    border-right: 1px solid #E2D9F7;
}

[data-testid="stSidebar"] > div:first-child {
    padding-top: 1.4rem;
}

[data-testid="stSidebar"] [data-testid="stMarkdownContainer"] p {
    color: #514967;
}

[data-testid="stSidebar"] [data-testid="stRadio"] > label {
    color: #49377A;
    font-weight: 700;
}

/* CARDS */

[data-testid="stVerticalBlockBorderWrapper"] {
    background: rgba(255, 255, 255, .94);
    border: 1px solid var(--border);
    border-radius: 20px;
    box-shadow: 0 7px 24px rgba(76, 49, 130, .045);
}

/* METRICS */

[data-testid="stMetric"] {
    background: linear-gradient(145deg, #FFFFFF, #F6F1FF);
    border: 1px solid #E5DDF7;
    padding: 20px;
    border-radius: 17px;
}

[data-testid="stMetricLabel"] {
    color: #756D8B !important;
}

[data-testid="stMetricValue"] {
    color: #6544BF !important;
}

/* INPUT FIELDS */

[data-testid="stTextInput"] label,
[data-testid="stSelectbox"] label,
[data-testid="stRadio"] > label {
    color: #43365F !important;
    font-weight: 600;
}

[data-testid="stTextInput"] input,
[data-testid="stTextArea"] textarea {
    background: #FFFFFF !important;
    color: #292044 !important;
    border: 1px solid #DCD3EE !important;
    border-radius: 11px !important;
}

[data-testid="stTextInput"] input:focus,
[data-testid="stTextArea"] textarea:focus {
    border-color: #8B6CE3 !important;
    box-shadow: 0 0 0 2px rgba(139, 108, 227, .13) !important;
}

[data-baseweb="select"] > div {
    background: #FFFFFF !important;
    border-color: #DCD3EE !important;
    border-radius: 11px !important;
    color: #292044 !important;
}

/* BUTTONS */

.stButton > button,
.stFormSubmitButton > button {
    color: #FFFFFF !important;
    background: linear-gradient(135deg, #8B6CE3, #6845C6);
    border: 1px solid #7959CF;
    border-radius: 12px;
    padding: .65rem 1rem;
    font-weight: 650;
    box-shadow: 0 4px 10px rgba(104, 69, 198, .16);
    transition: all .2s ease;
}

.stButton > button:hover,
.stFormSubmitButton > button:hover {
    background: linear-gradient(135deg, #7958D4, #5735B5);
    border-color: #5735B5;
    box-shadow: 0 7px 16px rgba(104, 69, 198, .24);
    transform: translateY(-2px);
}

.stButton > button:active,
.stFormSubmitButton > button:active {
    transform: translateY(0);
}

.stButton > button:focus,
.stFormSubmitButton > button:focus {
    box-shadow: 0 0 0 3px rgba(139, 108, 227, .2);
}

/* FORMS */

[data-testid="stForm"] {
    background: #FFFFFF;
    border: 1px solid #E8E0F5;
    border-radius: 16px;
    padding: 22px;
}

/* EXPANDERS */

[data-testid="stExpander"] {
    background: #FFFFFF;
    border: 1px solid #E5DDF4;
    border-radius: 13px;
}

[data-testid="stExpander"] summary {
    color: #5735B5;
    font-weight: 650;
}

/* TABLE */

[data-testid="stDataFrame"] {
    border: 1px solid #E6DFF5;
    border-radius: 12px;
    overflow: hidden;
}

/* ALERTS */

[data-testid="stAlert"] {
    border-radius: 12px;
}

/* CUSTOM COMPONENTS */

.brand {
    display: flex;
    align-items: center;
    gap: 12px;
    margin-bottom: 8px;
}

.brand-icon {
    width: 48px;
    height: 48px;
    border-radius: 15px;
    display: flex;
    align-items: center;
    justify-content: center;
    font-size: 24px;
    color: white;
    background: linear-gradient(135deg, #A58AEF, #704ACB);
    box-shadow: 0 6px 15px rgba(112, 74, 203, .2);
}

.brand-title {
    font-size: 24px;
    font-weight: 800;
    color: #30224F;
    letter-spacing: -.7px;
}

.brand-subtitle {
    font-size: 12px;
    color: #827797;
    margin-top: 2px;
}

.eyebrow {
    color: #7958CD;
    text-transform: uppercase;
    letter-spacing: 2px;
    font-size: 11px;
    font-weight: 750;
    margin-bottom: 12px;
}

.hero-title {
    color: #2F2449;
    font-size: clamp(30px, 4vw, 44px);
    line-height: 1.15;
    font-weight: 800;
    letter-spacing: -1.4px;
    margin-bottom: 13px;
}

.hero-description {
    color: #746A87;
    font-size: 15px;
    line-height: 1.8;
    max-width: 610px;
}

.pill {
    display: inline-block;
    padding: 7px 12px;
    border-radius: 999px;
    border: 1px solid #DED3FA;
    background: #F0EAFF;
    color: #6242B2;
    font-size: 12px;
    font-weight: 650;
    margin-right: 4px;
    margin-bottom: 5px;
}

.info-card {
    padding: 21px;
    border-radius: 17px;
    background: linear-gradient(145deg, #FFFFFF, #F5F0FF);
    border: 1px solid #E6DDF7;
    box-shadow: 0 6px 18px rgba(76, 49, 130, .04);
    min-height: 155px;
}

.info-card-title {
    font-size: 13px;
    color: #7C718F;
    margin-bottom: 9px;
}

.info-card-value {
    font-size: 24px;
    font-weight: 800;
    color: #6947C2;
    margin-bottom: 7px;
    overflow-wrap: anywhere;
}

.info-card-caption {
    color: #80758F;
    font-size: 12px;
    line-height: 1.6;
}

.section-label {
    font-size: 11px;
    letter-spacing: 1.6px;
    color: #896BD3;
    font-weight: 750;
    text-transform: uppercase;
    margin-bottom: 8px;
}

.footer-note {
    color: #958BA7;
    font-size: 12px;
    text-align: center;
    margin-top: 36px;
    padding-top: 18px;
    border-top: 1px solid #E9E2F5;
}

@media (max-width: 768px) {
    .block-container {
        padding: 1rem 1rem 2rem;
    }

    .hero-title {
        font-size: 30px;
    }

    .info-card {
        padding: 16px;
    }
}
</style>
""", unsafe_allow_html=True)


# ============================================================
# 3. REUSABLE UI
# ============================================================

def brand_header():
    st.markdown("""
    <div class="brand">
        <div class="brand-icon">🔐</div>
        <div>
            <div class="brand-title">SecureAuth</div>
            <div class="brand-subtitle">Your security. Your identity.</div>
        </div>
    </div>
    """, unsafe_allow_html=True)


def hero(eyebrow, title, description):
    st.markdown(
        f"""
        <div class="eyebrow">{eyebrow}</div>
        <div class="hero-title">{title}</div>
        <div class="hero-description">{description}</div>
        """,
        unsafe_allow_html=True
    )


def info_card(title, value, caption, symbol):
    st.markdown(
        f"""
        <div class="info-card">
            <div style="font-size:24px;margin-bottom:14px">{symbol}</div>
            <div class="info-card-title">{title}</div>
            <div class="info-card-value">{value}</div>
            <div class="info-card-caption">{caption}</div>
        </div>
        """,
        unsafe_allow_html=True
    )


def footer():
    st.markdown("""
    <div class="footer-note">
        SecureAuth · Authentication Portal
        <br>
        Built with Streamlit, SQLite, bcrypt and JWT
    </div>
    """, unsafe_allow_html=True)


# ============================================================
# 4. DATABASE
# ============================================================

def get_connection():
    return sqlite3.connect(DB_NAME, timeout=10)


def init_db():
    with get_connection() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS users (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                username TEXT NOT NULL COLLATE NOCASE UNIQUE,
                email TEXT NOT NULL COLLATE NOCASE UNIQUE,
                password_hash BLOB NOT NULL,
                security_question TEXT NOT NULL,
                security_answer_hash BLOB NOT NULL
            )
        """)


init_db()


def get_user_by_username(username):
    with get_connection() as conn:
        return conn.execute("""
            SELECT id, username, email, password_hash,
                   security_question, security_answer_hash
            FROM users
            WHERE username = ?
        """, (username.strip(),)).fetchone()


def get_user_by_email(email):
    with get_connection() as conn:
        return conn.execute("""
            SELECT id, username, email, password_hash,
                   security_question, security_answer_hash
            FROM users
            WHERE email = ?
        """, (email.strip(),)).fetchone()


# ============================================================
# 5. VALIDATION AND HASHING
# ============================================================

def validate_email(email):
    return bool(re.fullmatch(
        r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}",
        email.strip()
    ))


def validate_password(password):
    if len(password) < 8:
        return "Password must contain at least 8 characters."
    if len(password.encode("utf-8")) > 72:
        return "Password must not exceed 72 bytes."
    if not re.search(r"[A-Z]", password):
        return "Include at least one uppercase letter."
    if not re.search(r"[a-z]", password):
        return "Include at least one lowercase letter."
    if not re.search(r"\d", password):
        return "Include at least one digit."
    if not re.search(r"[^A-Za-z0-9]", password):
        return "Include at least one special character."
    return None


def hash_value(value):
    return bcrypt.hashpw(value.encode("utf-8"), bcrypt.gensalt())


def password_matches(value, stored_hash):
    if isinstance(stored_hash, str):
        stored_hash = stored_hash.encode("utf-8")

    try:
        return bcrypt.checkpw(value.encode("utf-8"), stored_hash)
    except (ValueError, TypeError):
        return False


def update_password(user_id, new_password):
    with get_connection() as conn:
        conn.execute(
            "UPDATE users SET password_hash = ? WHERE id = ?",
            (hash_value(new_password), user_id)
        )


# ============================================================
# 6. JWT AUTHENTICATION
# ============================================================

def issue_token(user_id, username):
    now = datetime.now(timezone.utc)

    return jwt.encode(
        {
            "user_id": user_id,
            "username": username,
            "iat": now,
            "exp": now + timedelta(hours=2)
        },
        JWT_SECRET,
        algorithm="HS256"
    )


def get_logged_in_user():
    token = st.session_state.get("auth_token")

    if not token:
        return None

    try:
        payload = jwt.decode(
            token,
            JWT_SECRET,
            algorithms=["HS256"]
        )
        return {
            "id": payload["user_id"],
            "username": payload["username"]
        }

    except jwt.PyJWTError:
        st.session_state.pop("auth_token", None)
        return None


# ============================================================
# 7. GMAIL OTP
# ============================================================

def send_otp_email(recipient, otp):
    if not EMAIL_ADDRESS or not EMAIL_PASSWORD:
        raise RuntimeError("Email credentials are not configured.")

    message = EmailMessage()
    message["Subject"] = "SecureAuth - Password Reset OTP"
    message["From"] = EMAIL_ADDRESS
    message["To"] = recipient
    message.set_content(
        f"""Hello,

Your SecureAuth password reset OTP is: {otp}

This OTP expires in 5 minutes.

If you did not request this reset, ignore this email.
"""
    )

    with smtplib.SMTP_SSL(
        "smtp.gmail.com", 465, timeout=20
    ) as server:
        server.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
        server.send_message(message)


def clear_otp_state():
    for key in (
        "reset_otp_hash",
        "reset_otp_expiry",
        "reset_email",
        "reset_otp_attempts"
    ):
        st.session_state.pop(key, None)


# ============================================================
# 8. SIGNUP PAGE
# ============================================================

def signup_page():
    brand_header()
    st.write("")

    left, right = st.columns([1.05, 1], gap="large")

    with left:
        hero(
            "CREATE YOUR ACCOUNT",
            "A safer space starts here.",
            "Create your account and access your personal workspace. "
            "Your password is protected using bcrypt hashing."
        )

        st.write("")
        info_card(
            "Password protection",
            "bcrypt",
            "Passwords are stored as secure hashes.",
            "🛡️"
        )

        st.write("")
        info_card(
            "Session security",
            "JWT",
            "Authenticated sessions expire after two hours.",
            "🔑"
        )

    with right:
        with st.container(border=True):
            st.markdown("### Create your account")
            st.caption("Enter your details to get started.")

            with st.form("signup_form"):
                username = st.text_input(
                    "Username",
                    placeholder="Choose a username"
                )
                email = st.text_input(
                    "Email address",
                    placeholder="you@example.com"
                )
                password = st.text_input(
                    "Password",
                    type="password",
                    placeholder="Create a strong password"
                )
                confirm_password = st.text_input(
                    "Confirm password",
                    type="password",
                    placeholder="Re-enter your password"
                )

                st.markdown("##### Account recovery")

                question = st.selectbox(
                    "Security question",
                    SECURITY_QUESTIONS
                )
                answer = st.text_input(
                    "Security answer",
                    type="password",
                    placeholder="Enter your answer"
                )

                submitted = st.form_submit_button(
                    "✨  Create account",
                    use_container_width=True
                )

            if submitted:
                username = username.strip()
                email = email.strip()

                if (
                    not username or not email or not password
                    or not confirm_password or not answer.strip()
                ):
                    st.error("Please complete all fields.")

                elif len(username) < 3:
                    st.error("Username must contain at least 3 characters.")

                elif not validate_email(email):
                    st.error("Please enter a valid email address.")

                elif validate_password(password):
                    st.error(validate_password(password))

                elif password != confirm_password:
                    st.error("Passwords do not match.")

                elif len(answer.strip()) < 2:
                    st.error("Please enter a valid security answer.")

                else:
                    try:
                        with get_connection() as conn:
                            conn.execute("""
                                INSERT INTO users (
                                    username, email, password_hash,
                                    security_question, security_answer_hash
                                )
                                VALUES (?, ?, ?, ?, ?)
                            """, (
                                username,
                                email,
                                hash_value(password),
                                question,
                                hash_value(answer.strip().casefold())
                            ))

                        st.success(
                            "Account created successfully! Open Login to sign in."
                        )

                    except sqlite3.IntegrityError:
                        st.error(
                            "That username or email is already registered."
                        )

    footer()


# ============================================================
# 9. PASSWORD RECOVERY
# ============================================================

def security_question_recovery():
    st.markdown("##### Verify your identity")
    st.caption("Use the security question selected during signup.")

    with st.form("security_recovery_form"):
        username = st.text_input(
            "Registered username",
            key="security_reset_username"
        )
        answer = st.text_input(
            "Security answer",
            type="password",
            key="security_reset_answer"
        )
        new_password = st.text_input(
            "New password",
            type="password",
            key="security_new_password"
        )
        confirm_password = st.text_input(
            "Confirm new password",
            type="password",
            key="security_confirm_password"
        )

        submitted = st.form_submit_button(
            "🔐  Reset password",
            use_container_width=True
        )

    if submitted:
        user = get_user_by_username(username)

        if not user or not password_matches(
            answer.strip().casefold(), user[5]
        ):
            st.error("Unable to verify the account and security answer.")

        elif validate_password(new_password):
            st.error(validate_password(new_password))

        elif new_password != confirm_password:
            st.error("Passwords do not match.")

        else:
            update_password(user[0], new_password)
            st.success("Password reset successfully. You can now log in.")


def email_otp_recovery():
    st.markdown("##### Verify your email")
    st.caption("Receive a one-time password at your registered email.")

    with st.form("request_otp_form"):
        email = st.text_input(
            "Registered email address",
            placeholder="you@example.com",
            key="otp_request_email"
        )

        request_submitted = st.form_submit_button(
            "✉️  Send email OTP",
            use_container_width=True
        )

    if request_submitted:
        email = email.strip()

        if not validate_email(email):
            st.error("Enter a valid email address.")

        else:
            user = get_user_by_email(email)

            if not user:
                st.info(
                    "If the address is registered, an OTP will be sent."
                )
            else:
                otp = f"{secrets.randbelow(1_000_000):06d}"

                try:
                    send_otp_email(user[2], otp)

                    st.session_state["reset_email"] = user[2]
                    st.session_state["reset_otp_hash"] = hash_value(otp)
                    st.session_state["reset_otp_expiry"] = (
                        datetime.now(timezone.utc)
                        + timedelta(minutes=5)
                    )
                    st.session_state["reset_otp_attempts"] = 0

                    st.success(
                        "If the address is registered, an OTP has been sent."
                    )

                except Exception:
                    st.error(
                        "Unable to send OTP. Check your email configuration."
                    )

    if st.session_state.get("reset_otp_hash"):
        st.info("Your OTP expires in 5 minutes.")

        with st.form("verify_otp_form"):
            entered_otp = st.text_input("6-digit OTP", max_chars=6)
            new_password = st.text_input(
                "New password",
                type="password",
                key="otp_new_password"
            )
            confirm_password = st.text_input(
                "Confirm new password",
                type="password",
                key="otp_confirm_password"
            )

            verify_submitted = st.form_submit_button(
                "🔐  Verify and reset password",
                use_container_width=True
            )

        if verify_submitted:
            expiry = st.session_state.get("reset_otp_expiry")
            attempts = st.session_state.get("reset_otp_attempts", 0)

            if not expiry or datetime.now(timezone.utc) >= expiry:
                clear_otp_state()
                st.error("OTP expired. Request a new one.")

            elif attempts >= 5:
                clear_otp_state()
                st.error("Maximum attempts reached. Request a new OTP.")

            else:
                st.session_state["reset_otp_attempts"] = attempts + 1

                if not re.fullmatch(r"\d{6}", entered_otp):
                    st.error("Enter a valid 6-digit OTP.")

                elif not password_matches(
                    entered_otp,
                    st.session_state["reset_otp_hash"]
                ):
                    if st.session_state["reset_otp_attempts"] >= 5:
                        clear_otp_state()
                        st.error(
                            "Maximum attempts reached. Request a new OTP."
                        )
                    else:
                        st.error("Incorrect OTP.")

                elif validate_password(new_password):
                    st.error(validate_password(new_password))

                elif new_password != confirm_password:
                    st.error("Passwords do not match.")

                else:
                    user = get_user_by_email(
                        st.session_state["reset_email"]
                    )

                    if user:
                        update_password(user[0], new_password)
                        clear_otp_state()
                        st.success(
                            "Password reset successfully. You can now log in."
                        )
                    else:
                        clear_otp_state()
                        st.error("Unable to complete the password reset.")


def forgot_password_section():
    with st.expander("🔑  Forgot your password?"):
        st.write(
            "Choose a recovery method to securely reset your account password."
        )

        method = st.radio(
            "Recovery method",
            ["Security question", "Email OTP"],
            horizontal=True,
            key="user_recovery_method"
        )

        st.divider()

        if method == "Security question":
            security_question_recovery()
        else:
            email_otp_recovery()


# ============================================================
# 10. LOGIN PAGE
# ============================================================

def login_page():
    left, right = st.columns([1.05, 1], gap="large")

    with left:
        brand_header()
        st.write("")
        st.write("")

        hero(
            "WELCOME TO SECUREAUTH",
            "Your security, made simple.",
            "Sign in to access your secure workspace. "
            "Your account is protected by hashed passwords "
            "and time-limited JWT sessions."
        )

        st.write("")
        st.markdown(
            '<span class="pill">✦ Secure authentication</span>'
            '<span class="pill">✦ Protected sessions</span>',
            unsafe_allow_html=True
        )

        st.write("")
        st.write("")

        info_card(
            "Your security",
            "Protected",
            "Authentication designed with security in mind.",
            "🛡️"
        )

    with right:
        st.write("")
        st.write("")

        with st.container(border=True):
            st.markdown("### Welcome back 👋")
            st.caption("Enter your credentials to continue.")

            with st.form("login_form"):
                username = st.text_input(
                    "Username",
                    placeholder="Your username"
                )
                password = st.text_input(
                    "Password",
                    type="password",
                    placeholder="Your password"
                )

                submitted = st.form_submit_button(
                    "✦  Sign in",
                    use_container_width=True
                )

            if submitted:
                user = get_user_by_username(username)

                if user and password_matches(password, user[3]):
                    st.session_state["auth_token"] = issue_token(
                        user[0], user[1]
                    )
                    st.rerun()
                else:
                    st.error("Invalid username or password.")

            st.write("")
            forgot_password_section()

    footer()


# ============================================================
# 11. USER DASHBOARD
# ============================================================

def user_dashboard(user):
    brand_header()
    st.write("")

    st.markdown(
        '<span class="pill">● Account authenticated</span>',
        unsafe_allow_html=True
    )

    hero(
        "YOUR PERSONAL WORKSPACE",
        f"Welcome back, {user['username']}.",
        "Your account is active. Here's an overview of your "
        "authentication status and account security."
    )

    st.write("")
    col1, col2, col3 = st.columns(3, gap="medium")

    with col1:
        info_card(
            "Account status",
            "Active",
            "You are successfully signed in.",
            "🟢"
        )

    with col2:
        info_card(
            "Authentication",
            "JWT Secured",
            "Your session uses a signed JWT.",
            "🔐"
        )

    with col3:
        info_card(
            "Session lifetime",
            "2 Hours",
            "Your token has a two-hour expiry.",
            "⏱️"
        )

    st.write("")

    with st.container(border=True):
        st.markdown(
            '<div class="section-label">ACCOUNT OVERVIEW</div>',
            unsafe_allow_html=True
        )
        st.markdown("### Your secure workspace")

        st.write(
            "You are authenticated and can use your account. "
            "Log out when you have finished using this session."
        )

        st.divider()

        detail1, detail2 = st.columns(2)

        with detail1:
            st.caption("SIGNED-IN USER")
            st.markdown(f"**{user['username']}**")

        with detail2:
            st.caption("SESSION SECURITY")
            st.markdown("**JWT · HS256 · 2-hour expiry**")

    st.write("")

    left, right = st.columns([3, 1])

    with left:
        st.caption("For your protection, log out on shared devices.")

    with right:
        if st.button("↪  Logout", use_container_width=True):
            st.session_state.pop("auth_token", None)
            st.rerun()

    footer()


# ============================================================
# 12. ADMIN LOGIN AND DASHBOARD
# ============================================================

def admin_login_page():
    brand_header()
    st.write("")

    if not st.session_state.get("admin_authenticated", False):
        left, right = st.columns([1.05, 1], gap="large")

        with left:
            hero(
                "RESTRICTED ACCESS",
                "Administrator portal.",
                "Manage registered accounts from your dedicated "
                "administrative workspace. This area is reserved "
                "for authorized administrators."
            )

            st.write("")
            info_card(
                "Access control",
                "Admin only",
                "A separate password protects this portal.",
                "🛡️"
            )

        with right:
            with st.container(border=True):
                st.markdown("### Admin sign in")
                st.caption("Authenticate to access the dashboard.")

                with st.form("admin_login_form"):
                    username = st.text_input(
                        "Admin username",
                        placeholder="Administrator username"
                    )
                    password = st.text_input(
                        "Admin password",
                        type="password",
                        placeholder="Administrator password"
                    )

                    submitted = st.form_submit_button(
                        "✦  Access dashboard",
                        use_container_width=True
                    )

                if submitted:
                    if not ADMIN_PASSWORD:
                        st.error(
                            "ADMIN_PASSWORD is missing from Colab Secrets."
                        )
                    elif (
                        secrets.compare_digest(
                            username, ADMIN_USERNAME
                        )
                        and secrets.compare_digest(
                            password, ADMIN_PASSWORD
                        )
                    ):
                        st.session_state["admin_authenticated"] = True
                        st.rerun()
                    else:
                        st.error("Invalid admin username or password.")

        footer()
        return

    # ---------------- ADMIN DASHBOARD ----------------

    header_left, header_right = st.columns([3, 1])

    with header_left:
        brand_header()

    with header_right:
        st.markdown(
            '<div style="text-align:right;padding-top:8px">'
            '<span class="pill">✦ Administrator</span></div>',
            unsafe_allow_html=True
        )

    st.write("")

    hero(
        "ADMINISTRATION",
        "Your control center.",
        "Review registered accounts and monitor your authentication portal."
    )

    with get_connection() as conn:
        users = conn.execute("""
            SELECT id, username, email
            FROM users
            ORDER BY id DESC
        """).fetchall()

        unique_emails = conn.execute("""
            SELECT COUNT(DISTINCT email)
            FROM users
        """).fetchone()[0]

    st.write("")
    col1, col2, col3 = st.columns(3, gap="medium")

    with col1:
        info_card(
            "Registered users",
            str(len(users)),
            "Total accounts in the database.",
            "👥"
        )

    with col2:
        info_card(
            "Unique email addresses",
            str(unique_emails),
            "Distinct registered email addresses.",
            "✉️"
        )

    with col3:
        info_card(
            "Portal status",
            "Operational",
            "Database is accessible.",
            "🟢"
        )

    st.write("")

    with st.container(border=True):
        st.markdown(
            '<div class="section-label">USER MANAGEMENT</div>',
            unsafe_allow_html=True
        )
        st.markdown("### Registered accounts")
        st.caption(
            "Overview of the accounts registered with SecureAuth."
        )

        st.divider()

        if users:
            st.dataframe(
                [
                    {
                        "User ID": user[0],
                        "Username": user[1],
                        "Email address": user[2],
                        "Status": "Registered"
                    }
                    for user in users
                ],
                use_container_width=True,
                hide_index=True
            )
        else:
            st.info("No users have registered yet.")

    st.write("")

    with st.container(border=True):
        st.markdown(
            '<div class="section-label">ADMINISTRATOR</div>',
            unsafe_allow_html=True
        )
        st.markdown("### Administrative session")
        st.write(
            "You are signed in to the administrator portal. "
            "Log out when you have finished reviewing account information."
        )

        if st.button("↪  Admin Logout", use_container_width=True):
            st.session_state["admin_authenticated"] = False
            st.rerun()

    footer()


# ============================================================
# 13. NAVIGATION
# ============================================================

st.sidebar.markdown("""
<div style="padding:5px 0 18px 0;">
    <div style="font-size:24px;font-weight:800;color:#30224F;">
        ◈ SecureAuth
    </div>
    <div style="font-size:12px;color:#827797;margin-top:5px;">
        LIGHT LAVENDER EDITION
    </div>
</div>
""", unsafe_allow_html=True)

st.sidebar.divider()

page = st.sidebar.radio(
    "NAVIGATION",
    ["Login", "Signup", "Admin Login"],
    key="main_navigation"
)

st.sidebar.divider()

st.sidebar.markdown("""
<div style="padding:8px 0;">
    <div style="font-size:11px;color:#7958CD;font-weight:750;
                letter-spacing:1.2px;">SECURITY FEATURES</div>
    <div style="margin-top:12px;font-size:13px;color:#514967;
                line-height:2.2;">
        🔐 &nbsp; bcrypt password hashing<br>
        🛡️ &nbsp; JWT authentication<br>
        ✉️ &nbsp; Email OTP recovery<br>
        🔑 &nbsp; Security question recovery
    </div>
</div>
""", unsafe_allow_html=True)

st.sidebar.markdown("""
<div style="margin-top:28px;padding:14px;
            border:1px solid #E2D8F5;
            border-radius:13px;background:#FFFFFF;">
    <div style="color:#5735B5;font-size:13px;font-weight:700;">
        SecureAuth Portal
    </div>
    <div style="color:#827797;font-size:11px;margin-top:5px;">
        Authentication demonstration
    </div>
</div>
""", unsafe_allow_html=True)


if page == "Login":
    logged_in_user = get_logged_in_user()

    if logged_in_user:
        user_dashboard(logged_in_user)
    else:
        login_page()

elif page == "Signup":
    signup_page()

elif page == "Admin Login":
    admin_login_page()


Overwriting app.py


In [17]:

!streamlit run app.py --server.port 8501 --server.headless true > streamlit.log 2>&1 &


In [18]:

from pyngrok import ngrok
from google.colab import userdata

ngrok.set_auth_token(userdata.get("NGROK_AUTHTOKEN"))

public_url = ngrok.connect(8501, "http").public_url
print("Open your application here:", public_url)


Open your application here: https://creation-underwent-immortal.ngrok-free.dev
